In [ ]:
import sys
import numpy as np
import tensorflow as tf
import tensorflow.keras as K
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [ ]:
ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

In [ ]:
layers = [4096, 2048, 1]
epochs = 50
act_func = tf.nn.relu
dropout = 0.5
input_dropout = 0.2
eta = 1e-5
norm = 'tanh'

In [ ]:
X_tr, X_val, _, _, y_tr, y_val, _, _ = load(norm=norm)
print("Training data shape:", X_tr.shape)
print("Validation data shape:", X_val.shape)
print("NaN in X_tr:", np.isnan(X_tr).any())
print("NaN in y_tr:", np.isnan(y_tr).any())
print("Inf in X_tr:", np.isinf(X_tr).any())
print("Inf in y_tr:", np.isinf(y_tr).any())

In [ ]:
model = Sequential()
for i in range(len(layers)):
    if i == 0:
        model.add(Dense(
            layers[i],
            input_shape=(X_tr.shape[1],),
            activation=act_func,
            kernel_initializer='he_normal'
        ))
        model.add(Dropout(float(input_dropout)))
    elif i == len(layers) - 1:
        model.add(Dense(
            layers[i],
            activation='linear',
            kernel_initializer="he_normal"
        ))
    else:
        model.add(Dense(
            layers[i],
            activation=act_func,
            kernel_initializer="he_normal"
        ))
        model.add(Dropout(float(dropout)))

In [ ]:
model.compile(
    loss='mean_squared_error',
    optimizer=K.optimizers.SGD(
        learning_rate=float(eta),
        momentum=0.5
    )
)
model.summary()

In [ ]:
hist = model.fit(
    X_tr, y_tr,
    epochs=epochs,
    batch_size=64,
    shuffle=True,
    validation_data=(X_val, y_val),
    verbose=1   
)

In [ ]:
val_loss = hist.history['val_loss']
train_loss = hist.history['loss']
print("Final training loss:", train_loss[-1])
print("Final validation loss:", val_loss[-1])
model.reset_states()